# ☠️ Notebook 5: Dead Letter Queue

Isolate and handle poison messages that repeatedly fail.

## Learning Objectives

By the end of this notebook, you'll understand:
- What dead letter queues are
- Why poison messages are dangerous
- How to implement DLQ
- DLQ monitoring and reprocessing

In [ ]:
import redis
import psycopg2
import json
import uuid
import time
import random
from datetime import datetime
from typing import Optional

r = redis.Redis(host='localhost', port=6379, decode_responses=True)

conn = psycopg2.connect(
    host="localhost", port=5432,
    database="taskqueue", user="postgres", password="postgres"
)
conn.autocommit = True

print("✅ Connected to Redis and PostgreSQL!")

## 🔄 Reset State

Run this cell to clear all jobs, logs, and queue data before running demos.

In [ ]:
def reset_all():
    r.flushall()
    cursor = conn.cursor()
    cursor.execute("DELETE FROM job_logs")
    cursor.execute("DELETE FROM dead_letter_queue")
    cursor.execute("DELETE FROM jobs")
    cursor.close()
    print("🔄 Reset complete: Redis flushed, jobs/logs/dlq cleared")

reset_all()

## ☠️ The Poison Message Problem

In [ ]:
print("☠️ What is a Poison Message?")
print("=" * 60)
print("""
A message that ALWAYS fails, no matter how many retries.

EXAMPLES:
• Corrupted data that can't be parsed
• References deleted resource
• Triggers unhandled exception
• Causes worker to crash

THE DOOM LOOP:
─────────────────────────────────────────────────────────────
     ┌──────────────────────────────────┐
     ▼                                  │
  [Queue] ──pop──> [Worker] ──fail──> [retry]
     ▲                                  │
     └──────────────────────────────────┘
     
Without DLQ, poison message loops forever:
• Wasting CPU cycles
• Blocking good messages
• Filling up logs
• Alerting on-call engineers repeatedly

THE SOLUTION - Dead Letter Queue:
─────────────────────────────────────────────────────────────
  [Main Queue]                    [Dead Letter Queue]
       │                                 │
       ▼                                 ▼
    [Worker]  ──max retries──>  [Isolation Area]
       │                                 │
       ▼                                 ▼
   [Success]                    [Manual Review]
""")

## 🔧 DLQ Implementation

In [ ]:
MAIN_QUEUE = "jobs:main"
DLQ = "jobs:dead"

class DLQManager:
    def __init__(self, conn, redis_client):
        self.conn = conn
        self.redis = redis_client
    
    def move_to_dlq(self, job_id: str, error: str, attempts: int):
        cursor = self.conn.cursor()
        cursor.execute("""
            INSERT INTO dead_letter_queue (job_id, error_message, attempts)
            VALUES (%s, %s, %s)
        """, (job_id, error, attempts))
        
        cursor.execute("""
            UPDATE jobs SET status = 'dead', error_message = %s WHERE id = %s
        """, (error, job_id))
        cursor.close()
        
        self.redis.lpush(DLQ, job_id)
        print(f"   ☠️ Moved {job_id[:8]}... to Dead Letter Queue")
    
    def get_dlq_count(self) -> int:
        return self.redis.llen(DLQ)
    
    def list_dlq_jobs(self, limit: int = 10):
        job_ids = self.redis.lrange(DLQ, 0, limit - 1)
        
        results = []
        cursor = self.conn.cursor()
        for job_id in job_ids:
            cursor.execute("""
                SELECT j.id, j.job_type, j.payload, d.error_message, d.attempts
                FROM dead_letter_queue d
                JOIN jobs j ON j.id = d.job_id
                WHERE d.job_id = %s
            """, (job_id,))
            row = cursor.fetchone()
            if row:
                results.append({
                    'id': row[0],
                    'type': row[1],
                    'payload': row[2],
                    'error': row[3],
                    'attempts': row[4]
                })
        cursor.close()
        return results
    
    def retry_from_dlq(self, job_id: str) -> bool:
        cursor = self.conn.cursor()
        cursor.execute("""
            UPDATE jobs SET status = 'pending', attempts = 0, error_message = NULL
            WHERE id = %s
        """, (job_id,))
        
        cursor.execute("DELETE FROM dead_letter_queue WHERE job_id = %s", (job_id,))
        cursor.close()
        
        self.redis.lrem(DLQ, 1, job_id)
        self.redis.lpush(MAIN_QUEUE, job_id)
        
        print(f"   ♻️ Requeued {job_id[:8]}... for retry")
        return True
    
    def delete_from_dlq(self, job_id: str) -> bool:
        cursor = self.conn.cursor()
        cursor.execute("DELETE FROM dead_letter_queue WHERE job_id = %s", (job_id,))
        cursor.close()
        
        self.redis.lrem(DLQ, 1, job_id)
        print(f"   🗑️ Deleted {job_id[:8]}... permanently")
        return True

dlq_manager = DLQManager(conn, r)
print("✅ DLQManager ready!")

In [ ]:
class WorkerWithDLQ:
    def __init__(self, conn, redis_client, max_attempts: int = 3):
        self.conn = conn
        self.redis = redis_client
        self.max_attempts = max_attempts
        self.dlq = DLQManager(conn, redis_client)
    
    def process_job(self, job_id: str, handler) -> str:
        cursor = self.conn.cursor()
        cursor.execute("SELECT attempts FROM jobs WHERE id = %s", (job_id,))
        row = cursor.fetchone()
        attempts = (row[0] if row else 0) + 1
        
        cursor.execute("""
            UPDATE jobs SET status = 'processing', attempts = %s WHERE id = %s
        """, (attempts, job_id))
        cursor.close()
        
        try:
            result = handler()
            
            cursor = self.conn.cursor()
            cursor.execute("""
                UPDATE jobs SET status = 'completed', result = %s WHERE id = %s
            """, (json.dumps(result), job_id))
            cursor.close()
            return 'completed'
            
        except Exception as e:
            if attempts >= self.max_attempts:
                self.dlq.move_to_dlq(job_id, str(e), attempts)
                return 'dead'
            else:
                cursor = self.conn.cursor()
                cursor.execute("""
                    UPDATE jobs SET status = 'pending', error_message = %s WHERE id = %s
                """, (str(e), job_id))
                cursor.close()
                
                self.redis.lpush(MAIN_QUEUE, job_id)
                print(f"   🔄 Retry {attempts}/{self.max_attempts}")
                return 'retry'

print("✅ WorkerWithDLQ ready!")

In [ ]:
print("☠️ Dead Letter Queue Demo")
print("=" * 60)

def create_test_job(job_type: str, will_fail: bool) -> str:
    job_id = str(uuid.uuid4())
    cursor = conn.cursor()
    cursor.execute("""
        INSERT INTO jobs (id, job_type, payload, status, attempts)
        VALUES (%s, %s, %s, 'pending', 0)
    """, (job_id, job_type, json.dumps({'will_fail': will_fail})))
    cursor.close()
    r.lpush(MAIN_QUEUE, job_id)
    return job_id

print("\n1️⃣ Creating test jobs...")
good_job = create_test_job('report', will_fail=False)
print(f"   Good job: {good_job[:8]}...")

poison_job = create_test_job('report', will_fail=True)
print(f"   Poison job: {poison_job[:8]}...")

print(f"\n📊 Queue length: {r.llen(MAIN_QUEUE)}")

In [ ]:
print("\n2️⃣ Processing jobs...")
print("=" * 60)

worker = WorkerWithDLQ(conn, r, max_attempts=3)

cursor = conn.cursor()
poison_jobs = {}
cursor.execute("SELECT id, payload FROM jobs WHERE status = 'pending'")
for row in cursor.fetchall():
    payload = json.loads(row[1]) if row[1] else {}
    poison_jobs[row[0]] = payload.get('will_fail', False)
cursor.close()

processed = 0
while True:
    result = r.brpop(MAIN_QUEUE, timeout=1)
    if not result:
        break
    
    job_id = result[1]
    is_poison = poison_jobs.get(job_id, False)
    
    print(f"\n   Processing {job_id[:8]}...")
    
    def handler():
        if is_poison:
            raise Exception("Corrupted data: cannot parse payload")
        return {"status": "success"}
    
    status = worker.process_job(job_id, handler)
    
    if status == 'completed':
        print(f"   ✅ Completed")
    elif status == 'dead':
        print(f"   ☠️ Sent to DLQ")
    
    processed += 1
    if processed > 10:
        break

print(f"\n📊 Total processed: {processed}")

In [ ]:
print("\n3️⃣ DLQ Inspection")
print("=" * 60)

print(f"\n📊 DLQ Count: {dlq_manager.get_dlq_count()}")

dlq_jobs = dlq_manager.list_dlq_jobs()
print(f"\n📋 Dead Jobs:")
for job in dlq_jobs:
    print(f"\n   ID: {job['id'][:8]}...")
    print(f"   Type: {job['type']}")
    print(f"   Error: {job['error']}")
    print(f"   Attempts: {job['attempts']}")

## 🔧 Manual Intervention

In [ ]:
print("🔧 Manual Intervention Options")
print("=" * 60)
print("""
When a job lands in DLQ, you have three choices:

1. FIX AND RETRY
   • Fix the bug in code
   • Fix corrupted data
   • Requeue the job

2. DELETE PERMANENTLY
   • Job is no longer needed
   • Data is irrecoverable
   • Manual compensation done

3. KEEP FOR ANALYSIS
   • Debug why it failed
   • Collect statistics
   • Train ML models on failures
""")

if dlq_jobs:
    test_job = dlq_jobs[0]
    print(f"\n🔄 Retrying job {test_job['id'][:8]}...")
    dlq_manager.retry_from_dlq(test_job['id'])
    
    print(f"\n📊 Queue after retry: {r.llen(MAIN_QUEUE)}")
    print(f"📊 DLQ after retry: {dlq_manager.get_dlq_count()}")

## 📊 DLQ Monitoring Dashboard

In [ ]:
def dlq_dashboard():
    print("📊 DLQ Monitoring Dashboard")
    print("=" * 60)
    
    cursor = conn.cursor()
    
    cursor.execute("""
        SELECT COUNT(*), 
               AVG(attempts) as avg_attempts,
               MIN(created_at) as oldest,
               MAX(created_at) as newest
        FROM dead_letter_queue
    """)
    stats = cursor.fetchone()
    
    print(f"\n📈 Overview:")
    print(f"   Total dead jobs: {stats[0]}")
    print(f"   Avg attempts: {stats[1]:.1f}" if stats[1] else "   Avg attempts: N/A")
    
    cursor.execute("""
        SELECT j.job_type, COUNT(*) as count
        FROM dead_letter_queue d
        JOIN jobs j ON j.id = d.job_id
        GROUP BY j.job_type
        ORDER BY count DESC
    """)
    
    print(f"\n📊 By Job Type:")
    for row in cursor.fetchall():
        print(f"   {row[0]}: {row[1]}")
    
    cursor.execute("""
        SELECT error_message, COUNT(*) as count
        FROM dead_letter_queue
        GROUP BY error_message
        ORDER BY count DESC
        LIMIT 5
    """)
    
    print(f"\n🔥 Top Errors:")
    for row in cursor.fetchall():
        error_short = row[0][:50] + "..." if len(row[0]) > 50 else row[0]
        print(f"   {row[1]}x: {error_short}")
    
    cursor.close()

dlq_dashboard()

## 🧪 Quick Quiz

1. **What's a poison message?**

2. **What happens without a DLQ?**

3. **What are the options for handling DLQ jobs?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Poison message:")
print("   - A message that always fails")
print("   - No matter how many retries")
print("   - Examples: corrupted data, deleted resource")
print()
print("2. Without DLQ:")
print("   - Infinite retry loop")
print("   - Wasted resources")
print("   - Good jobs get blocked")
print("   - Log spam")
print()
print("3. DLQ handling options:")
print("   - Fix and retry")
print("   - Delete permanently")
print("   - Keep for analysis")

## 📚 Summary

### Key Takeaways

1. **Poison messages** always fail, cause doom loops
2. **DLQ isolates** problematic jobs after max retries
3. **Manual review** - fix, delete, or analyze
4. **Monitor DLQ** - alerts, dashboards, trends
5. **Don't ignore** - DLQ growth indicates problems

### Next Up

In **Notebook 6**, we'll explore advanced patterns:
- Idempotency keys
- Backpressure
- Priority queues